In [ ]:
from google.colab import files
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))


Saving calovibe.ipynb to calovibe.ipynb
Saving fitness_app.db to fitness_app (1).db
Saving model_utils.py to model_utils (1).py
Saving requirements.txt to requirements.txt
Saving train_model.py to train_model (1).py
Saving app.py to app (7).py
Saving Calories Burned During Exercise Dataset.zip to Calories Burned During Exercise Dataset (7).zip
Saving Fitness_Nutrition_App.ipynb to Fitness_Nutrition_App (7).ipynb
Saving Gym Exercises Dataset.zip to Gym Exercises Dataset (7).zip
Saving Indian Food Nutritional Values Dataset.zip to Indian Food Nutritional Values Dataset (7).zip
Saving prepare_data.py to prepare_data (7).py
Saving USDA FoodData Central : Food Nutrition & Calories Dataset.zip to USDA FoodData Central : Food Nutrition & Calories Dataset (7).zip
Saving Indian_Food_Nutrition_Processed.csv to Indian_Food_Nutrition_Processed (7).csv
Saving gym_members_exercise_tracking.csv to gym_members_exercise_tracking (7).csv
Saving Gym Exercises Dataset.xlsx to Gym Exercises Dataset (7).xls

In [ ]:
%%writefile prepare_data.py
"""
prepare_data.py
----------------
Consolidates all raw datasets (USDA foods, Indian foods, branded/allergen foods,
gym exercise library, and gym calorie-burn tracking logs) into a single SQLite
database (fitness_app.db) that the Streamlit app reads from.

Run this once after uploading the raw files to the Colab session:
    python prepare_data.py
"""

import glob
import os
import shutil
import sqlite3
import zipfile

import numpy as np
import pandas as pd

RAW_DIR = "data_raw"
DB_PATH = "fitness_app.db"


# ---------------------------------------------------------------------------
# 0. Unpack everything into a flat raw-data folder
# ---------------------------------------------------------------------------
def stage_raw_files():
    os.makedirs(RAW_DIR, exist_ok=True)

    for z in glob.glob("*.zip"):
        with zipfile.ZipFile(z) as zf:
            zf.extractall(RAW_DIR)
        print(f"Unzipped {z}")

    loose_files = [
        "comprehensive_foods_usda.csv",
        "foods_allergens.csv",
        "foods_dietary_restrictions.csv",
        "foods_health_scores_allergens.csv",
        "healthy_foods_database.csv",
    ]
    for f in loose_files:
        if os.path.exists(f):
            dest = os.path.join(RAW_DIR, f)
            if not os.path.exists(dest):
                shutil.copy(f, dest)


def find_file(substr):
    """Find the first file under RAW_DIR whose name contains substr (case-insensitive)."""
    substr = substr.lower()
    for root, _, files in os.walk(RAW_DIR):
        for f in files:
            if substr in f.lower():
                return os.path.join(root, f)
    return None


# ---------------------------------------------------------------------------
# 1. Helper: simple 0-100 health score heuristic for datasets that lack one
# ---------------------------------------------------------------------------
def heuristic_health_score(protein, fiber, sugar, sodium_mg, sat_fat=0):
    protein = pd.to_numeric(protein, errors="coerce").fillna(0)
    fiber = pd.to_numeric(fiber, errors="coerce").fillna(0)
    sugar = pd.to_numeric(sugar, errors="coerce").fillna(0)
    sodium_mg = pd.to_numeric(sodium_mg, errors="coerce").fillna(0)
    sat_fat = pd.to_numeric(sat_fat, errors="coerce").fillna(0) if not isinstance(sat_fat, int) else 0
    score = 50 + protein * 1.5 + fiber * 2 - sugar * 0.7 - (sodium_mg / 100.0) - sat_fat * 1.0
    return score.clip(0, 100).round(1)


NUTRISCORE_MAP = {"a": 90, "b": 75, "c": 60, "d": 40, "e": 20}

FOOD_COLUMNS = [
    "name", "source", "food_type", "calories", "protein_g", "carbs_g", "fat_g",
    "fiber_g", "sugar_g", "sodium_mg", "health_score",
    "contains_gluten", "contains_dairy", "contains_nuts", "contains_soy",
    "contains_eggs", "contains_fish", "serving_note",
]


def empty_food_frame():
    return pd.DataFrame(columns=FOOD_COLUMNS)


# ---------------------------------------------------------------------------
# 2. Load + standardize each food source
# ---------------------------------------------------------------------------
def load_usda():
    path = find_file("comprehensive_foods_usda")
    if not path:
        return empty_food_frame()
    df = pd.read_csv(path, low_memory=False)
    out = pd.DataFrame({
        "name": df["food_name"],
        "source": "USDA",
        "food_type": df.get("food_category", df.get("food_type", "Unknown")).fillna("Unknown"),
        "calories": pd.to_numeric(df["calories"], errors="coerce"),
        "protein_g": pd.to_numeric(df["protein_g"], errors="coerce"),
        "carbs_g": pd.to_numeric(df["carbs_g"], errors="coerce"),
        "fat_g": pd.to_numeric(df["fat_g"], errors="coerce"),
        "fiber_g": pd.to_numeric(df["fiber_g"], errors="coerce"),
        "sugar_g": pd.to_numeric(df["sugar_g"], errors="coerce"),
        "sodium_mg": pd.to_numeric(df["sodium_mg"], errors="coerce"),
        "health_score": pd.to_numeric(df.get("health_score"), errors="coerce"),
        "contains_gluten": np.nan,
        "contains_dairy": np.nan,
        "contains_nuts": np.nan,
        "contains_soy": np.nan,
        "contains_eggs": np.nan,
        "contains_fish": np.nan,
        "serving_note": df.get("household_serving", "").fillna(""),
    })
    out["health_score"] = out["health_score"].fillna(
        heuristic_health_score(out["protein_g"], out["fiber_g"], out["sugar_g"], out["sodium_mg"])
    )
    return out


def load_indian():
    path = find_file("Indian_Food_Nutrition_Processed")
    if not path:
        return empty_food_frame()
    df = pd.read_csv(path)
    out = pd.DataFrame({
        "name": df["Dish Name"],
        "source": "Indian",
        "food_type": "Indian Dish",
        "calories": pd.to_numeric(df["Calories (kcal)"], errors="coerce"),
        "protein_g": pd.to_numeric(df["Protein (g)"], errors="coerce"),
        "carbs_g": pd.to_numeric(df["Carbohydrates (g)"], errors="coerce"),
        "fat_g": pd.to_numeric(df["Fats (g)"], errors="coerce"),
        "fiber_g": pd.to_numeric(df["Fibre (g)"], errors="coerce"),
        "sugar_g": pd.to_numeric(df["Free Sugar (g)"], errors="coerce"),
        "sodium_mg": pd.to_numeric(df["Sodium (mg)"], errors="coerce"),
        "health_score": np.nan,
        "contains_gluten": np.nan,
        "contains_dairy": np.nan,
        "contains_nuts": np.nan,
        "contains_soy": np.nan,
        "contains_eggs": np.nan,
        "contains_fish": np.nan,
        "serving_note": "1 serving",
    })
    out["health_score"] = heuristic_health_score(
        out["protein_g"], out["fiber_g"], out["sugar_g"], out["sodium_mg"]
    )
    return out


def load_branded():
    path = find_file("foods_health_scores_allergens")
    if not path:
        return empty_food_frame()
    df = pd.read_csv(path, low_memory=False)

    def score_row(grade, protein, fiber, sugar, sodium, satfat):
        g = str(grade).strip().lower()
        if g in NUTRISCORE_MAP:
            return NUTRISCORE_MAP[g]
        return None

    ns_scores = df["nutriscore_grade"].astype(str).str.strip().str.lower().map(NUTRISCORE_MAP)

    out = pd.DataFrame({
        "name": df["product_name"],
        "source": "Branded",
        "food_type": df.get("food_type", "Branded/Packaged").fillna("Branded/Packaged"),
        "calories": pd.to_numeric(df["energy_kcal"], errors="coerce"),
        "protein_g": pd.to_numeric(df["proteins_100g"], errors="coerce"),
        "carbs_g": pd.to_numeric(df["carbs_100g"], errors="coerce"),
        "fat_g": pd.to_numeric(df["fat_100g"], errors="coerce"),
        "fiber_g": pd.to_numeric(df["fiber_100g"], errors="coerce"),
        "sugar_g": pd.to_numeric(df["sugars_100g"], errors="coerce"),
        "sodium_mg": pd.to_numeric(df["sodium_100g"], errors="coerce") * 1000,  # g -> mg
        "health_score": ns_scores,
        "contains_gluten": df.get("contains_gluten"),
        "contains_dairy": df.get("contains_dairy"),
        "contains_nuts": df.get("contains_nuts"),
        "contains_soy": df.get("contains_soy"),
        "contains_eggs": df.get("contains_eggs"),
        "contains_fish": df.get("contains_fish"),
        "serving_note": "per 100g",
    })
    out["health_score"] = out["health_score"].fillna(
        heuristic_health_score(out["protein_g"], out["fiber_g"], out["sugar_g"], out["sodium_mg"])
    )
    return out


def build_foods_table():
    frames = [load_usda(), load_indian(), load_branded()]
    foods = pd.concat(frames, ignore_index=True, sort=False)
    foods = foods.dropna(subset=["name", "calories"])
    foods = foods[foods["calories"] >= 0]
    for col in ["contains_gluten", "contains_dairy", "contains_nuts",
                "contains_soy", "contains_eggs", "contains_fish"]:
        foods[col] = foods[col].astype("object")  # keep NaN = "unknown" distinct from False
    foods = foods.reset_index(drop=True)
    foods.insert(0, "food_id", foods.index + 1)
    return foods


# ---------------------------------------------------------------------------
# 3. Exercise calorie-burn coefficients (calories per minute per kg body weight)
# ---------------------------------------------------------------------------
def build_workout_coefficients():
    path = find_file("gym_members_exercise_tracking")
    if not path:
        return pd.DataFrame(columns=["workout_type", "cal_per_min_per_kg", "n_samples"])
    df = pd.read_csv(path)
    minutes = df["Session_Duration (hours)"] * 60
    df["cal_per_min_per_kg"] = df["Calories_Burned"] / minutes / df["Weight (kg)"]
    df = df[np.isfinite(df["cal_per_min_per_kg"])]
    grouped = (
        df.groupby("Workout_Type")["cal_per_min_per_kg"]
        .agg(["mean", "count"])
        .reset_index()
        .rename(columns={"Workout_Type": "workout_type", "mean": "cal_per_min_per_kg", "count": "n_samples"})
    )
    grouped["cal_per_min_per_kg"] = grouped["cal_per_min_per_kg"].round(4)
    return grouped


# ---------------------------------------------------------------------------
# 4. Exercise library (browsable gym exercises)
# ---------------------------------------------------------------------------
def build_exercise_library():
    path = find_file("Gym Exercises Dataset")
    if not path:
        path = find_file("Gym_Exercises_Dataset")
    if not path or not path.lower().endswith((".xlsx", ".xls")):
        # search generically for an xlsx under raw dir
        for root, _, files in os.walk(RAW_DIR):
            for f in files:
                if f.lower().endswith(".xlsx"):
                    path = os.path.join(root, f)
    if not path:
        return pd.DataFrame(columns=["exercise_name", "muscle_group", "equipment", "rating", "description", "url"])

    df = pd.read_excel(path)
    out = pd.DataFrame({
        "exercise_name": df["Exercise_Name"],
        "muscle_group": df["muscle_gp"],
        "equipment": df["Equipment"],
        "rating": df["Rating"],
        "description": df["Description"],
        "url": df.get("Description_URL", ""),
    })
    out = out.dropna(subset=["exercise_name"]).reset_index(drop=True)
    out.insert(0, "exercise_id", out.index + 1)
    return out


# ---------------------------------------------------------------------------
# 5. Main
# ---------------------------------------------------------------------------
def main():
    print("Staging raw files...")
    stage_raw_files()

    print("Building foods table (USDA + Indian + Branded)...")
    foods = build_foods_table()
    print(f"  -> {len(foods):,} food items")

    print("Building workout calorie-burn coefficients...")
    coeffs = build_workout_coefficients()
    print(f"  -> {len(coeffs)} workout types")

    print("Building exercise library...")
    exercises = build_exercise_library()
    print(f"  -> {len(exercises):,} exercises")

    if os.path.exists(DB_PATH):
        os.remove(DB_PATH)

    with sqlite3.connect(DB_PATH) as conn:
        foods.to_sql("foods", conn, index=False)
        coeffs.to_sql("workout_coefficients", conn, index=False)
        exercises.to_sql("exercise_library", conn, index=False)
        conn.execute("CREATE INDEX idx_foods_name ON foods(name)")
        conn.execute("CREATE INDEX idx_ex_muscle ON exercise_library(muscle_group)")

    print(f"\nDone. Wrote {DB_PATH}")


if __name__ == "__main__":
    main()


Overwriting prepare_data.py


In [ ]:
!python prepare_data.py


Staging raw files...
Unzipped Indian Food Nutritional Values Dataset (5).zip
Unzipped Gym Exercises Dataset (3).zip
Unzipped USDA FoodData Central : Food Nutrition & Calories Dataset (3).zip
Unzipped Calories Burned During Exercise Dataset (1).zip
Unzipped Indian Food Nutritional Values Dataset (1).zip
Unzipped Gym Exercises Dataset (6).zip
Unzipped USDA FoodData Central : Food Nutrition & Calories Dataset (4).zip
Unzipped Calories Burned During Exercise Dataset (5).zip
Unzipped Gym Exercises Dataset (4).zip
Unzipped Indian Food Nutritional Values Dataset (6).zip
Unzipped Calories Burned During Exercise Dataset (2).zip
Unzipped USDA FoodData Central : Food Nutrition & Calories Dataset (5).zip
Unzipped USDA FoodData Central : Food Nutrition & Calories Dataset.zip
Unzipped Gym Exercises Dataset (2).zip
Unzipped Indian Food Nutritional Values Dataset (4).zip
Unzipped Calories Burned During Exercise Dataset (7).zip
Unzipped USDA FoodData Central : Food Nutrition & Calories Dataset (2).zip


In [ ]:
%%writefile app.py
"""
app.py - CALOVIBE
------------------------------------------------
Run with:  streamlit run app.py
Requires fitness_app.db to already exist (created by prepare_data.py).
"""

import sqlite3
import time
from datetime import datetime

import pandas as pd
import streamlit as st

DB_PATH = "fitness_app.db"

st.set_page_config(page_title="CALOVIBE", page_icon="🏋️", layout="wide")


# ---------------------------------------------------------------------------
# Data access
# ---------------------------------------------------------------------------
@st.cache_resource
def get_conn():
    return sqlite3.connect(DB_PATH, check_same_thread=False)


@st.cache_data
def load_foods():
    return pd.read_sql("SELECT * FROM foods", get_conn())


@st.cache_data
def load_coefficients():
    return pd.read_sql("SELECT * FROM workout_coefficients", get_conn())


@st.cache_data
def load_exercises():
    return pd.read_sql("SELECT * FROM exercise_library", get_conn())


foods_df = load_foods()
coeff_df = load_coefficients()
exercise_df = load_exercises()


# ---------------------------------------------------------------------------
# Session state (per-user logs, in-memory for the session)
# ---------------------------------------------------------------------------
if "food_log" not in st.session_state:
    st.session_state.food_log = []
if "workout_log" not in st.session_state:
    st.session_state.workout_log = []


# ---------------------------------------------------------------------------
# Sidebar: profile
# ---------------------------------------------------------------------------
st.sidebar.header("👤 Your Profile")
age = st.sidebar.number_input("Age", 10, 100, 28)
sex = st.sidebar.selectbox("Sex", ["Male", "Female"])
height_cm = st.sidebar.number_input("Height (cm)", 100.0, 250.0, 170.0)
weight_kg = st.sidebar.number_input("Weight (kg)", 30.0, 250.0, 70.0)
activity = st.sidebar.selectbox(
    "Activity level",
    ["Sedentary", "Lightly active", "Moderately active", "Very active", "Extremely active"],
)
goal = st.sidebar.selectbox("Goal", ["Lose weight", "Maintain weight", "Gain weight"])

ACTIVITY_FACTORS = {
    "Sedentary": 1.2,
    "Lightly active": 1.375,
    "Moderately active": 1.55,
    "Very active": 1.725,
    "Extremely active": 1.9,
}
GOAL_ADJUST = {"Lose weight": -500, "Maintain weight": 0, "Gain weight": 300}

# Mifflin-St Jeor BMR
if sex == "Male":
    bmr = 10 * weight_kg + 6.25 * height_cm - 5 * age + 5
else:
    bmr = 10 * weight_kg + 6.25 * height_cm - 5 * age - 161

tdee = bmr * ACTIVITY_FACTORS[activity]
target_calories = max(1200, tdee + GOAL_ADJUST[goal])
bmi = weight_kg / ((height_cm / 100) ** 2)

if bmi < 18.5:
    bmi_cat = "Underweight"
elif bmi < 25:
    bmi_cat = "Normal"
elif bmi < 30:
    bmi_cat = "Overweight"
else:
    bmi_cat = "Obese"

st.sidebar.markdown("---")
st.sidebar.metric("BMI", f"{bmi:.1f}", bmi_cat)
st.sidebar.metric("TDEE (maintenance)", f"{tdee:.0f} kcal/day")
st.sidebar.metric("Daily calorie target", f"{target_calories:.0f} kcal")


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def totals_today():
    eaten = sum(item["calories"] for item in st.session_state.food_log)
    burned = sum(item["calories_burned"] for item in st.session_state.workout_log)
    return eaten, burned


def allergen_filter(df, exclude_gluten, exclude_dairy, exclude_nuts, exclude_soy, exclude_eggs, exclude_fish):
    out = df.copy()
    checks = {
        "contains_gluten": exclude_gluten,
        "contains_dairy": exclude_dairy,
        "contains_nuts": exclude_nuts,
        "contains_soy": exclude_soy,
        "contains_eggs": exclude_eggs,
        "contains_fish": exclude_fish,
    }
    for col, exclude in checks.items():
        if exclude:
            # keep rows where flag is explicitly False; drop True. Unknown (NaN) is kept.
            out = out[out[col] != True]  # noqa: E712
    return out


st.title("🏋️‍♀️ All-in-One Fitness & Nutrition Tracker")

tab_dash, tab_food, tab_workout, tab_reco = st.tabs(
    ["📊 Dashboard", "🍽️ Food Search & Log", "💪 Workouts & Calories Burned", "✨ Recommendations"]
)


# ---------------------------------------------------------------------------
# TAB: Dashboard
# ---------------------------------------------------------------------------
with tab_dash:
    eaten, burned = totals_today()
    net = eaten - burned
    remaining = target_calories - net

    c1, c2, c3, c4 = st.columns(4)
    c1.metric("Calories eaten", f"{eaten:.0f}")
    c2.metric("Calories burned", f"{burned:.0f}")
    c3.metric("Net calories", f"{net:.0f}")
    c4.metric("Remaining budget", f"{remaining:.0f}")

    progress = min(max(net / target_calories, 0), 1) if target_calories else 0
    st.progress(progress, text=f"{net:.0f} / {target_calories:.0f} kcal target")

    col_a, col_b = st.columns(2)
    with col_a:
        st.subheader("Today's meals")
        if st.session_state.food_log:
            fl = pd.DataFrame(st.session_state.food_log)
            st.dataframe(fl[["name", "servings", "calories", "protein_g", "carbs_g", "fat_g"]],
                         use_container_width=True, hide_index=True)
            if st.button("Clear food log"):
                st.session_state.food_log = []
                st.rerun()
            st.bar_chart(fl.set_index("name")[["protein_g", "carbs_g", "fat_g"]])
        else:
            st.info("No meals logged yet. Add some from the Food tab.")

    with col_b:
        st.subheader("Today's workouts")
        if st.session_state.workout_log:
            wl = pd.DataFrame(st.session_state.workout_log)
            st.dataframe(wl[["workout_type", "duration_min", "calories_burned"]],
                         use_container_width=True, hide_index=True)
            if st.button("Clear workout log"):
                st.session_state.workout_log = []
                st.rerun()
        else:
            st.info("No workouts logged yet. Add some from the Workouts tab.")


# ---------------------------------------------------------------------------
# TAB: Food Search & Log
# ---------------------------------------------------------------------------
with tab_food:
    st.subheader("Search the food database")
    st.caption(f"{len(foods_df):,} items from USDA, Indian dishes, and branded/packaged foods.")

    colf1, colf2, colf3 = st.columns([2, 1, 1])
    with colf1:
        query = st.text_input("Search by name", "")
    with colf2:
        source_filter = st.multiselect("Source", sorted(foods_df["source"].unique()),
                                        default=list(foods_df["source"].unique()))
    with colf3:
        max_cal = st.slider("Max calories per item", 0, 900, 900)

    with st.expander("Dietary / allergen filters"):
        e1, e2, e3, e4, e5, e6 = st.columns(6)
        exclude_gluten = e1.checkbox("Exclude gluten")
        exclude_dairy = e2.checkbox("Exclude dairy")
        exclude_nuts = e3.checkbox("Exclude nuts")
        exclude_soy = e4.checkbox("Exclude soy")
        exclude_eggs = e5.checkbox("Exclude eggs")
        exclude_fish = e6.checkbox("Exclude fish")

    results = foods_df[foods_df["source"].isin(source_filter)]
    if query:
        results = results[results["name"].str.contains(query, case=False, na=False)]
    results = results[results["calories"] <= max_cal]
    results = allergen_filter(results, exclude_gluten, exclude_dairy, exclude_nuts,
                               exclude_soy, exclude_eggs, exclude_fish)
    results = results.sort_values("health_score", ascending=False).head(200)

    st.dataframe(
        results[["name", "source", "calories", "protein_g", "carbs_g", "fat_g",
                 "fiber_g", "sugar_g", "sodium_mg", "health_score", "serving_note"]],
        use_container_width=True, hide_index=True, height=320,
    )

    st.markdown("##### Log a food")
    if len(results) > 0:
        pick_name = st.selectbox("Pick an item from results above", results["name"].tolist())
        servings = st.number_input("Servings", 0.25, 10.0, 1.0, step=0.25)
        if st.button("➕ Add to today's log"):
            row = results[results["name"] == pick_name].iloc[0]
            st.session_state.food_log.append({
                "name": row["name"],
                "servings": servings,
                "calories": round(row["calories"] * servings, 1),
                "protein_g": round((row["protein_g"] or 0) * servings, 1),
                "carbs_g": round((row["carbs_g"] or 0) * servings, 1),
                "fat_g": round((row["fat_g"] or 0) * servings, 1),
                "time": datetime.now().strftime("%H:%M"),
            })
            st.success(f"Added {pick_name} ({servings} serving(s))")
    else:
        st.warning("No foods match your filters.")


# ---------------------------------------------------------------------------
# TAB: Workouts
# ---------------------------------------------------------------------------
with tab_workout:
    left, right = st.columns([1, 1])

    with left:
        st.subheader("Calorie-burn estimator")
        st.caption("Coefficients learned from real gym member session data.")
        workout_type = st.selectbox("Workout type", coeff_df["workout_type"].tolist())
        duration_min = st.slider("Duration (minutes)", 5, 180, 30)
        coef = coeff_df.loc[coeff_df["workout_type"] == workout_type, "cal_per_min_per_kg"].iloc[0]
        est_calories = coef * duration_min * weight_kg
        st.metric("Estimated calories burned", f"{est_calories:.0f} kcal")

        if st.button("➕ Log this workout"):
            st.session_state.workout_log.append({
                "workout_type": workout_type,
                "duration_min": duration_min,
                "calories_burned": round(est_calories, 1),
                "time": datetime.now().strftime("%H:%M"),
            })
            st.success(f"Logged {workout_type} ({duration_min} min)")

    with right:
        st.subheader("Exercise library")
        muscle_groups = ["All"] + sorted(exercise_df["muscle_group"].dropna().unique().tolist())
        mg = st.selectbox("Muscle group", muscle_groups)
        ex_query = st.text_input("Search exercise name", "")

        ex_results = exercise_df.copy()
        if mg != "All":
            ex_results = ex_results[ex_results["muscle_group"] == mg]
        if ex_query:
            ex_results = ex_results[ex_results["exercise_name"].str.contains(ex_query, case=False, na=False)]

        st.dataframe(
            ex_results[["exercise_name", "muscle_group", "equipment", "rating"]].head(100),
            use_container_width=True, hide_index=True, height=320,
        )


# ---------------------------------------------------------------------------
# TAB: Recommendations
# ---------------------------------------------------------------------------
with tab_reco:
    eaten, burned = totals_today()
    remaining = max(target_calories - eaten + burned, 0)
    st.subheader(f"Suggested foods (fit within ~{remaining/3:.0f} kcal per meal)")

    reco_foods = allergen_filter(foods_df, exclude_gluten, exclude_dairy, exclude_nuts,
                                  exclude_soy, exclude_eggs, exclude_fish) \
        if any([exclude_gluten, exclude_dairy, exclude_nuts, exclude_soy, exclude_eggs, exclude_fish]) \
        else foods_df
    meal_budget = max(remaining / 3, 100)
    reco_foods = reco_foods[reco_foods["calories"] <= meal_budget]
    reco_foods = reco_foods.sort_values("health_score", ascending=False).head(10)
    st.dataframe(
        reco_foods[["name", "source", "calories", "protein_g", "health_score"]],
        use_container_width=True, hide_index=True,
    )

    st.subheader("Suggested workout focus")
    if goal == "Lose weight":
        suggested = ["HIIT", "Cardio"]
        muscle_focus = ["Abdominals", "Quadriceps", "Glutes"]
    elif goal == "Gain weight":
        suggested = ["Strength"]
        muscle_focus = ["Chest", "Lats", "Quadriceps", "Biceps", "Triceps"]
    else:
        suggested = ["Yoga", "Strength", "Cardio"]
        muscle_focus = ["Abdominals", "Chest", "Lats"]

    best = coeff_df[coeff_df["workout_type"].isin(suggested)].sort_values(
        "cal_per_min_per_kg", ascending=False
    )
    st.write(f"Recommended workout types for **{goal}**: " + ", ".join(suggested))
    st.dataframe(best, use_container_width=True, hide_index=True)

    st.write("Matching exercises to try:")
    reco_ex = exercise_df[exercise_df["muscle_group"].isin(muscle_focus)].sample(
        min(8, len(exercise_df[exercise_df["muscle_group"].isin(muscle_focus)]))
    )
    st.dataframe(reco_ex[["exercise_name", "muscle_group", "equipment", "rating"]],
                 use_container_width=True, hide_index=True)

st.caption("⚠️ Estimates are for general fitness tracking, not medical advice.")


Overwriting app.py


In [ ]:
!pip install streamlit

In [ ]:
import time, subprocess, re, os

# Download cloudflared if we don't already have it
if not os.path.exists('cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared

# Make sure any previous run isn't still holding the port
!pkill -f streamlit 2>/dev/null
!pkill -f cloudflared 2>/dev/null
time.sleep(2)

# Start the Streamlit app
get_ipython().system_raw('streamlit run app.py --server.port 8501 --server.headless true > streamlit.log 2>&1 &')
time.sleep(6)

# Start the Cloudflare quick tunnel pointed at it
get_ipython().system_raw('./cloudflared tunnel --url http://localhost:8501 > cloudflared.log 2>&1 &')

# Poll the log for the public URL (usually appears within ~10-15s)
url = None
for _ in range(20):
    time.sleep(2)
    if os.path.exists('cloudflared.log'):
        text = open('cloudflared.log').read()
        match = re.search(r'https://[a-zA-Z0-9.-]*\.trycloudflare\.com', text)
        if match:
            url = match.group(0)
            break

if url:
    print('Your app is live at:', url)
else:
    print('No URL found yet — re-run this cell, or check streamlit.log / cloudflared.log below.')
    print(open('cloudflared.log').read() if os.path.exists('cloudflared.log') else 'no cloudflared.log yet')

^C
^C
Your app is live at: https://grill-listings-slight-temp.trycloudflare.com


In [ ]:
print(open('streamlit.log').read())



2026-09-11 04:16:11.024 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.185.87.204:8501




In [ ]:
# Cell 1 — upload everything
from google.colab import files
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

Saving calovibe.ipynb to calovibe (1).ipynb
Saving fitness_app.db to fitness_app (2).db
Saving model_utils.py to model_utils (2).py
Saving requirements.txt to requirements (1).txt
Saving train_model.py to train_model (2).py
Saving app.py to app (8).py
Saving Calories Burned During Exercise Dataset.zip to Calories Burned During Exercise Dataset (8).zip
Saving Fitness_Nutrition_App.ipynb to Fitness_Nutrition_App (8).ipynb
Saving Gym Exercises Dataset.zip to Gym Exercises Dataset (8).zip
Saving Indian Food Nutritional Values Dataset.zip to Indian Food Nutritional Values Dataset (8).zip
Saving prepare_data.py to prepare_data (8).py
Saving USDA FoodData Central : Food Nutrition & Calories Dataset.zip to USDA FoodData Central : Food Nutrition & Calories Dataset (8).zip
Saving Indian_Food_Nutrition_Processed.csv to Indian_Food_Nutrition_Processed (8).csv
Saving gym_members_exercise_tracking.csv to gym_members_exercise_tracking (8).csv
Saving Gym Exercises Dataset.xlsx to Gym Exercises Dataset

In [ ]:
# Cell 2 — install deps
!pip install -q streamlit scikit-learn joblib openpyxl

In [ ]:
# Cell 3 — write model_utils.py
%%writefile model_utils.py
# (paste the full model_utils.py content here)

Overwriting model_utils.py


In [ ]:
# Cell 4 — write prepare_data.py
%%writefile prepare_data.py
# (paste the full prepare_data.py content here)

Overwriting prepare_data.py


In [ ]:
# Cell 5 — write train_model.py
%%writefile train_model.py
# (paste the full train_model.py content here)

Overwriting train_model.py


In [ ]:
# Cell 6 — write app.py
%%writefile app.py
# (paste the full app.py content here)

Overwriting app.py


In [ ]:
# Cell 7 — run the pipeline
!python prepare_data.py
!python train_model.py

In [ ]:
# Cell 8 — launch + tunnel (cloudflared)
import time, subprocess, re, os
if not os.path.exists('cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared
get_ipython().system_raw('streamlit run app.py --server.port 8501 &')
time.sleep(5)
get_ipython().system_raw('./cloudflared tunnel --url http://localhost:8501 > cloudflared.log 2>&1 &')
time.sleep(6)
log = open('cloudflared.log').read()
url = re.search(r'https://[a-zA-Z0-9\-]+\.trycloudflare\.com', log)
print('App URL:', url.group(0) if url else 'check cloudflared.log')

App URL: https://liquid-faster-steps-logged.trycloudflare.com
